In [9]:
from osgeo import gdal
import helper
import os
import numpy as np

In [10]:
stressed = helper.getStressedImagesNames('stress_date.xlsx')

In [11]:
stressed[0]

(0, 'geojson_0-2018-10-01-2018-10-11-S2.tif')

In [12]:
ROOT_DIR = r"SWIFTT_dataset_France"
OUTPUT_DIR = r"dataset_cropped_64_coniferous"
DIM = 64

driver = gdal.GetDriverByName("GTiff")

for group, img_name in stressed:
    img = gdal.Open(os.path.join(ROOT_DIR, str(group), img_name))
    mask = gdal.Open(os.path.join(ROOT_DIR, str(group), f"geojson_{group}_mask.tif"))
    ftype = gdal.Open(os.path.join(ROOT_DIR, str(group), f"geojson_{group}_forest_type_2018.tif"))

    stepsCountX = img.RasterXSize // DIM
    stepsCountY = img.RasterYSize // DIM
    # stepX = img.RasterXSize // stepsCountX
    # stepY = img.RasterYSize // stepsCountY
    for i in range(0, (stepsCountX + 1) * DIM, DIM):
        for j in range(0, (stepsCountY + 1) * DIM, DIM):
            if i + DIM > img.RasterXSize or j + DIM > img.RasterYSize:
                continue
            print(i + DIM, img.RasterXSize, j + DIM, img.RasterYSize)

            ftype_mask = ftype.GetRasterBand(1).ReadAsArray(i, j, DIM, DIM)
            ftype_mask = (ftype_mask == 2).astype("int8")

            data = mask.GetRasterBand(1).ReadAsArray(i, j, DIM, DIM) * ftype_mask
            uniq, coun = np.unique(data, return_counts=True)
            if len(uniq) == 1 or np.min(coun) < 16:
                continue

            # img
            output_path = os.path.join(OUTPUT_DIR, f"g{group}_xoff_{i}_yoff_{j}_S2.tif")
            out_dataset = driver.Create(output_path, DIM, DIM, img.RasterCount, img.GetRasterBand(1).DataType)
            
            for k in range(img.RasterCount):
                out_dataset.GetRasterBand(k+1).WriteArray(img.GetRasterBand(k+1).ReadAsArray(i, j, DIM, DIM) * ftype_mask)

            geotransform = img.GetGeoTransform()
            out_dataset.SetGeoTransform((geotransform[0] + i * geotransform[1], 
                                         geotransform[1], 0, 
                                         geotransform[3] + j * geotransform[5], 
                                         0, geotransform[5]))
            out_dataset.SetProjection(img.GetProjection())
            out_dataset = None

            # label
            output_path = os.path.join(OUTPUT_DIR, f"g{group}_xoff_{i}_yoff_{j}_LABEL.tif")
            out_dataset = driver.Create(output_path, DIM, DIM, mask.RasterCount, gdal.GDT_Byte)
            
            for k in range(mask.RasterCount):
                out_dataset.GetRasterBand(k+1).WriteArray(mask.GetRasterBand(k+1).ReadAsArray(i, j, DIM, DIM) * ftype_mask)

            geotransform = mask.GetGeoTransform()
            out_dataset.SetGeoTransform((geotransform[0] + i * geotransform[1], 
                                         geotransform[1], 0, 
                                         geotransform[3] + j * geotransform[5], 
                                         0, geotransform[5]))
            out_dataset.SetProjection(mask.GetProjection())
            out_dataset = None

            # mask
            output_path = os.path.join(OUTPUT_DIR, f"g{group}_xoff_{i}_yoff_{j}_MASK.tif")
            out_dataset = driver.Create(output_path, DIM, DIM, mask.RasterCount, gdal.GDT_Byte)
            
            for k in range(mask.RasterCount):
                out_dataset.GetRasterBand(k+1).WriteArray(255 * mask.GetRasterBand(k+1).ReadAsArray(i, j, DIM, DIM) * ftype_mask)

            geotransform = mask.GetGeoTransform()
            out_dataset.SetGeoTransform((geotransform[0] + i * geotransform[1], 
                                         geotransform[1], 0, 
                                         geotransform[3] + j * geotransform[5], 
                                         0, geotransform[5]))
            out_dataset.SetProjection(mask.GetProjection())
            out_dataset = None

64 71 64 65
64 131 64 93
128 131 64 93
64 70 64 71
64 110 64 86
64 123 64 120
64 103 64 74
64 100 64 99
64 152 64 154
64 152 128 154
128 152 64 154
128 152 128 154
64 72 64 66
64 113 64 120
64 71 64 94
64 110 64 107
64 75 64 78
64 81 64 76
64 108 64 112
64 346 64 194
64 346 128 194
64 346 192 194
128 346 64 194
128 346 128 194
128 346 192 194
192 346 64 194
192 346 128 194
192 346 192 194
256 346 64 194
256 346 128 194
256 346 192 194
320 346 64 194
320 346 128 194
320 346 192 194
64 566 64 330
64 566 128 330
64 566 192 330
64 566 256 330
64 566 320 330
128 566 64 330
128 566 128 330
128 566 192 330
128 566 256 330
128 566 320 330
192 566 64 330
192 566 128 330
192 566 192 330
192 566 256 330
192 566 320 330
256 566 64 330
256 566 128 330
256 566 192 330
256 566 256 330
256 566 320 330
320 566 64 330
320 566 128 330
320 566 192 330
320 566 256 330
320 566 320 330
384 566 64 330
384 566 128 330
384 566 192 330
384 566 256 330
384 566 320 330
448 566 64 330
448 566 128 330
448 566 192 33